# Parte 3 — Extração dos 17 laudos com IA

Este notebook parte do **texto original dos 17 laudos**, envia cada documento à Gemini e produz um JSON com os mesmos campos. Depois valida a estrutura, confere as evidências literais e compara as respostas a uma referência revisável. Ele pode ser executado no **VS Code**

**Entrada:** `data/raw/laudos/laudo_01.txt` a `laudo_17.txt` (também aceita `laudos_avaliacao/` e o sufixo `(1)` dos primeiros arquivos). **Saída:** `outputs/parte3_ia/`, com JSONs individuais, JSONL, CSV de comparação e resumo. Os laudos de terreno **entram** nesta etapa; a exclusão do CSV da Parte 1 não se aplica aqui.

**Ordem:** execute todas as células de cima para baixo. A chamada real à API será feita somente nas células identificadas como tal. Consulte a [documentação de saídas estruturadas](https://ai.google.dev/gemini-api/docs/structured-output) e a [documentação do SDK](https://ai.google.dev/gemini-api/docs/get-started).


## 1. Dependências e chave

A chave é solicitada pelo kernel e não é salva no `.ipynb`. Se reiniciar o kernel, execute novamente. O pacote `pandas` é usado apenas para exibir/exportar a comparação.


In [2]:
%pip install -U google-genai 'pydantic>=2,<3' pandas


Note: you may need to restart the kernel to use updated packages.


O sistema n�o pode encontrar o arquivo especificado.


In [3]:
import getpass
import os

if not (os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')):
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Cole sua chave Gemini: ')
assert os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'), 'Chave vazia.'
print('Chave disponível neste kernel.')


Chave disponível neste kernel.


## 2. Localização dos arquivos

São necessários os **17** arquivos. Se faltar algum, a execução para com uma mensagem mostrando as pastas verificadas. Os arquivos de origem permanecem intactos.


In [4]:
from pathlib import Path

RAIZES = [Path.cwd(), Path.cwd().parent]
SUBPASTAS = [Path('data/raw/laudos'), Path('laudos_avaliacao')]
CANDIDATAS = [raiz / sub for raiz in RAIZES for sub in SUBPASTAS]

def localizar(pasta, n):
    nome = f'laudo_{n:02d}.txt'
    direto = pasta / nome
    duplicado = pasta / f'laudo_{n:02d}(1).txt'
    return direto if direto.is_file() else (duplicado if duplicado.is_file() else None)

PASTA_LAUDOS = next((p for p in CANDIDATAS if all(localizar(p, n) for n in range(1, 18))), None)
if PASTA_LAUDOS is None:
    raise FileNotFoundError(f'Não encontrei 17 laudos juntos. Verifique: {[str(p) for p in CANDIDATAS]}')

ARQUIVOS = {f'laudo_{n:02d}.txt': localizar(PASTA_LAUDOS, n) for n in range(1, 18)}
TEXTOS = {nome: path.read_text(encoding='utf-8-sig') for nome, path in ARQUIVOS.items()}
RAIZ_PROJETO = PASTA_LAUDOS.parents[2] if PASTA_LAUDOS.parts[-3:] == ('data', 'raw', 'laudos') else PASTA_LAUDOS.parent
SAIDA = RAIZ_PROJETO / 'outputs' / 'parte3_ia'
print('Pasta dos laudos:', PASTA_LAUDOS.resolve())
print('Laudos localizados:', len(TEXTOS))
print('Saídas em:', SAIDA.resolve())


Pasta dos laudos: C:\ProjetosDev\case_bari\data\raw\laudos
Laudos localizados: 17
Saídas em: C:\ProjetosDev\case_bari\outputs\parte3_ia


## 3. Contrato JSON

Todas as chaves são obrigatórias. `null` significa ausência, não aplicabilidade ou dúvida **explicada nos alertas**. Áreas são uma lista tipada em m²; no caso de contradição, `valor_m2` fica `null` e os números aparecem em `valores_conflitantes_m2`. A situação de ônus distingue gravame, cancelamento, declaração sem certidão e falta de informação. Datas de vistoria seguem `AAAA-MM-DD`.

O esquema Pydantic é enviado à API como `response_format`, e a resposta é validada **de novo** no Python. Isso garante formato e tipos, mas não garante veracidade dos valores; as próximas células fazem verificações adicionais.


In [5]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

class Estrito(BaseModel):
    model_config = ConfigDict(extra='forbid')

class Area(Estrito):
    tipo: Literal['privativa', 'total', 'construida', 'terreno', 'util', 'comum', 'coberta', 'outra']
    valor_m2: float | None
    valores_conflitantes_m2: list[float]
    texto_original: str

class Onus(Estrito):
    situacao: Literal['sem_onus_identificados', 'onus_identificados', 'onus_cancelado',
                     'sem_onus_declarados_sem_certidao', 'nao_informado', 'contraditorio']
    descricao: str | None

class Evidencias(Estrito):
    tipo_imovel: list[str]
    endereco: list[str]
    areas: list[str]
    ano_construcao: list[str]
    valor_avaliacao_brl: list[str]
    matricula: list[str]
    onus: list[str]
    data_vistoria: list[str]
    responsavel_tecnico: list[str]

class Alerta(Estrito):
    campo: Literal['tipo_imovel', 'endereco', 'areas', 'ano_construcao',
                   'valor_avaliacao_brl', 'matricula', 'onus', 'data_vistoria',
                   'responsavel_tecnico']
    tipo: Literal['ausente', 'nao_aplicavel', 'ambiguidade', 'contraditorio', 'revisao_manual']
    detalhe: str

class Laudo(Estrito):
    arquivo: str
    tipo_imovel: str | None
    endereco: str | None
    areas: list[Area]
    ano_construcao: int | None
    valor_avaliacao_brl: float | None
    matricula: str | None
    onus: Onus
    data_vistoria: str | None
    responsavel_tecnico: str | None
    evidencias: Evidencias
    alertas: list[Alerta]

ESQUEMA = Laudo.model_json_schema()
print('Contrato:', list(Laudo.model_fields), '| campos obrigatórios:', len(ESQUEMA['required']))


Contrato: ['arquivo', 'tipo_imovel', 'endereco', 'areas', 'ano_construcao', 'valor_avaliacao_brl', 'matricula', 'onus', 'data_vistoria', 'responsavel_tecnico', 'evidencias', 'alertas'] | campos obrigatórios: 12


## 4. Instruções genéricas de extração

O mesmo prompt vale para todos os arquivos. Ele não contém respostas dos laudos nem conhece o gabarito. Converter hectares para m² não muda o tipo da área. Evidências devem ser trechos **literais** do documento.


In [6]:
REGRAS = '''Extraia somente fatos do laudo abaixo. Responda de acordo com o esquema JSON fornecido.
Regras:
1. Preserve o nome do arquivo informado. Não invente endereço, matrícula, área, ano ou data.
2. Campos ausentes recebem null e alerta 'ausente'; ano inaplicável recebe null e 'nao_aplicavel'.
3. Se um ano for apenas idade aparente, ano de referência ou 'Ano' sem explicação, ano_construcao=null e alerta 'ambiguidade'.
4. Cada área é distinta (privativa, total, construída, terreno, útil, comum, coberta). Converta ha em m² multiplicando por 10000. Números positivos; para conflito de valores do mesmo tipo, valor_m2=null, liste TODOS os valores em valores_conflitantes_m2 e crie alerta 'contraditorio'.
5. Ônus: 'sem_onus_identificados' apenas quando o documento relatar negativa de certidão; 'onus_identificados' para gravame/restrição registrada; 'onus_cancelado' para registro cancelado; 'sem_onus_declarados_sem_certidao' para declaração não comprovada; 'nao_informado' para ausência de informação ou impossibilidade de verificação. Descrição preserva ressalvas.
6. Data da vistoria/inspeção/visita técnica em AAAA-MM-DD. Data de levantamento isolada não comprova vistoria: null com alerta. Data de documento consultado não substitui data de vistoria.
7. Em evidencias, use trechos copiados literalmente do laudo para cada campo. Se o campo é null por ambiguidade, use o trecho que causou a dúvida. Em areas, texto_original também deve ser trecho literal.
8. Se o texto se contradiz, NÃO escolha um número arbitrariamente. Preserve candidatos e alerta. Não afirme situação jurídica atual além do texto.
'''

def montar_prompt(nome, texto):
    return f'{REGRAS}\nARQUIVO: {nome}\nLAUDO:\n{texto}'


## 5. Chamada com saída estruturada e checagens locais

Em caso de erro de cota (`429`), fazemos até duas novas tentativas com espera curta. Erros restantes são registrados como falhas; **não** se fabrica JSON. O validador local exige trechos de evidência presentes no arquivo, números positivos, datas reais e alertas em situações de dúvida.


In [7]:
import json
import time
from datetime import date
from google import genai

MODELO = 'gemini-3.5-flash-lite'
cliente = genai.Client()

def extrair(nome, texto):
    for tentativa in range(3):
        try:
            resposta = cliente.interactions.create(
                model=MODELO,
                input=montar_prompt(nome, texto),
                response_format={'type': 'text', 'mime_type': 'application/json', 'schema': ESQUEMA},
                store=False,
            )
            if not resposta.output_text:
                raise ValueError('API retornou texto vazio')
            item = Laudo.model_validate_json(resposta.output_text)
            if item.arquivo != nome:
                raise ValueError(f'Nome de origem inesperado: {item.arquivo!r}')
            return item
        except Exception as erro:
            if '429' in str(erro) and tentativa < 2:
                time.sleep(4 * (tentativa + 1))
                continue
            raise

def auditar(item, texto):
    problemas = []
    campos = list(Evidencias.model_fields)
    for campo in campos:
        trechos = getattr(item.evidencias, campo)
        for trecho in trechos:
            if not trecho.strip() or trecho not in texto:
                problemas.append(f'{campo}: evidência não encontrada literalmente')
        valor = getattr(item, campo)
        if campo not in ('areas', 'onus') and valor is not None and not trechos:
            problemas.append(f'{campo}: valor preenchido sem evidência')
        if campo not in ('areas', 'onus') and valor is None:
            if not any(a.campo == campo and a.tipo in ('ausente','nao_aplicavel','ambiguidade','contraditorio') for a in item.alertas):
                problemas.append(f'{campo}: null sem alerta explicativo')
    for area in item.areas:
        if not area.texto_original.strip() or area.texto_original not in texto:
            problemas.append('areas: texto_original não aparece no laudo')
        if area.valor_m2 is None:
            if len(area.valores_conflitantes_m2) < 2 or not any(a.campo == 'areas' and a.tipo == 'contraditorio' for a in item.alertas):
                problemas.append('areas: contradição sem dois candidatos e alerta')
        elif area.valor_m2 <= 0 or area.valores_conflitantes_m2:
            problemas.append('areas: valor não positivo ou candidatos indevidos')
        if any(v <= 0 for v in area.valores_conflitantes_m2):
            problemas.append('areas: candidato não positivo')
    if item.areas and not item.evidencias.areas:
        problemas.append('areas: faltam evidências')
    if item.onus.situacao != 'nao_informado' and not item.evidencias.onus:
        problemas.append('onus: classificação sem evidência')
    if item.ano_construcao is not None and not (1800 <= item.ano_construcao <= 2025):
        problemas.append('ano_construcao: fora do intervalo plausível dos laudos')
    if item.valor_avaliacao_brl is not None and item.valor_avaliacao_brl <= 0:
        problemas.append('valor_avaliacao_brl: não positivo')
    if item.data_vistoria is not None:
        try:
            if date.fromisoformat(item.data_vistoria).isoformat() != item.data_vistoria:
                problemas.append('data_vistoria: formato fora de AAAA-MM-DD')
        except ValueError:
            problemas.append('data_vistoria: data inválida')
    if item.onus.situacao == 'contraditorio' and not any(a.campo == 'onus' and a.tipo == 'contraditorio' for a in item.alertas):
        problemas.append('onus: contradição sem alerta')
    return problemas


## 6. Teste de um laudo (primeira chamada à API)

Confira o JSON e as mensagens de auditoria antes de seguir para o lote. Esse teste será reaproveitado para não cobrar uma segunda chamada pelo mesmo laudo ao rodar a célula seguinte.


In [8]:
resultados = {}
falhas = {}
nome_teste = 'laudo_01.txt'
resultados[nome_teste] = extrair(nome_teste, TEXTOS[nome_teste])
print(resultados[nome_teste].model_dump_json(indent=2))
print('Auditoria:', auditar(resultados[nome_teste], TEXTOS[nome_teste]) or 'sem problemas automáticos')


{
  "arquivo": "laudo_01.txt",
  "tipo_imovel": "apartamento residencial",
  "endereco": "Rua das Acácias, 145, ap. 82 - Vila Mariana, São Paulo/SP",
  "areas": [
    {
      "tipo": "privativa",
      "valor_m2": 78.4,
      "valores_conflitantes_m2": [],
      "texto_original": "Área privativa: 78,40 m²"
    },
    {
      "tipo": "total",
      "valor_m2": 102.1,
      "valores_conflitantes_m2": [],
      "texto_original": "área total: 102,10 m²"
    }
  ],
  "ano_construcao": 2014,
  "valor_avaliacao_brl": 642000.0,
  "matricula": "184.772 do 14º CRI de São Paulo",
  "onus": {
    "situacao": "sem_onus_identificados",
    "descricao": "não foram identificados ônus na certidão analisada."
  },
  "data_vistoria": "2025-03-12",
  "responsavel_tecnico": "Eng. Marina Albuquerque - CREA-SP 5061234567.",
  "evidencias": {
    "tipo_imovel": [
      "apartamento residencial"
    ],
    "endereco": [
      "Rua das Acácias, 145, ap. 82 - Vila Mariana, São Paulo/SP"
    ],
    "areas": [
   

## 7. Rodar os 17 e salvar os resultados

O lote continua quando um laudo falha e registra o erro, sem inventar um documento. Reexecutar a célula na mesma sessão **não repete** solicitações já concluídas. Os JSONs são gravados mesmo quando a auditoria aponta problemas, com as pendências separadas no resumo para revisão.


In [9]:
import pandas as pd

SAIDA.mkdir(parents=True, exist_ok=True)
for nome, texto in TEXTOS.items():
    if nome in resultados:
        continue
    try:
        resultados[nome] = extrair(nome, texto)
        falhas.pop(nome, None)
        print(nome, 'OK')
    except Exception as erro:
        falhas[nome] = f'{type(erro).__name__}: {erro}'
        print(nome, 'FALHOU:', type(erro).__name__)

auditorias = {nome: auditar(item, TEXTOS[nome]) for nome, item in resultados.items()}
for nome, item in resultados.items():
    (SAIDA / nome.replace('.txt', '.json')).write_text(item.model_dump_json(indent=2) + '\n', encoding='utf-8')
(SAIDA / 'laudos_extraidos.jsonl').write_text(
    ''.join(resultados[n].model_dump_json() + '\n' for n in sorted(resultados)), encoding='utf-8'
)
(SAIDA / 'execucao.json').write_text(json.dumps({
    'modelo': MODELO, 'esperados': 17, 'extraidos': len(resultados),
    'falhas': falhas, 'auditoria': auditorias
}, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Concluídos: {len(resultados)}/17 | Falhas: {len(falhas)} | Pendências de auditoria: {sum(bool(v) for v in auditorias.values())}')
if falhas:
    print('Arquivos com falha:', ', '.join(falhas))


laudo_02.txt OK
laudo_03.txt OK
laudo_04.txt OK
laudo_05.txt OK
laudo_06.txt OK
laudo_07.txt OK
laudo_08.txt OK
laudo_09.txt OK
laudo_10.txt OK
laudo_11.txt OK
laudo_12.txt OK
laudo_13.txt OK
laudo_14.txt OK
laudo_15.txt OK
laudo_16.txt OK
laudo_17.txt OK
Concluídos: 17/17 | Falhas: 0 | Pendências de auditoria: 5


## 8. Referência para avaliação — revise antes de apresentar como gabarito humano

A tabela abaixo foi **preparada com auxílio de IA a partir dos 17 textos** e serve como referência inicial para testar o código. **Ela não é um gabarito humano independente.** Leia cada `.txt`, altere o que discordar e documente suas decisões; só depois trate as métricas como avaliação revisada. Não edite a referência para coincidir com as respostas do modelo.

Critério de acerto: nove campos por documento (153 verificações). Texto e matrícula são comparados após remover diferenças de maiúsculas, acentos, espaços e pontuação. Números usam tolerância de R$ 0,01 e 0,01 m²; datas ISO e situações de ônus exigem igualdade. Áreas devem coincidir em **tipo, valor e candidatos de contradição**. A descrição livre de ônus e a semântica do endereço exigem revisão humana adicional, porque igualdade textual pode rejeitar equivalências corretas.


In [10]:
def ref(tipo, endereco, areas, ano, valor, matricula, onus, data, tecnico):
    return dict(tipo_imovel=tipo, endereco=endereco, areas=areas,
                ano_construcao=ano, valor_avaliacao_brl=valor, matricula=matricula,
                onus=onus, data_vistoria=data, responsavel_tecnico=tecnico)

# Área esperada: (tipo, valor_m2 ou None, candidatos de conflito).
GABARITO_RASCUNHO = {
 'laudo_01.txt': ref('apartamento residencial','Rua das Acácias, 145, ap. 82 - Vila Mariana, São Paulo/SP', [('privativa',78.4,()),('total',102.1,())],2014,642000,'184.772','sem_onus_identificados','2025-03-12','Eng. Marina Albuquerque - CREA-SP 5061234567'),
 'laudo_02.txt': ref('casa térrea','Av. Central, 900, bloco B, Belo Horizonte - MG', [('construida',146,()),('terreno',250,())],2008,680000,'45.981','sem_onus_identificados','2025-03-18','Carlos Henrique Moura, CAU A123456-7'),
 'laudo_03.txt': ref('sala comercial 503','Edifício Horizonte, Rua do Comércio, 77, Curitiba/PR', [('util',54.8,()),('comum',18.2,())],None,395500,'77.201','onus_identificados','2025-03-22','Arq. Beatriz Nunes (CAU A987654-3)'),
 'laudo_04.txt': ref('terreno urbano','Lote 18, Quadra F, Rua Ipê Amarelo, Goiânia/GO', [('terreno',360,())],None,218000,'102.334','nao_informado','2025-04-02','Paulo Sérgio Reis, CREA 12345/D-GO'),
 'laudo_05.txt': ref('imóvel rural denominado Sítio Boa Vista','Município de Campinas, UF SP', [('terreno',48000,()),('construida',310,())],1999,1275000,'32.110','onus_identificados','2025-04-07','João A. Farias, CREA-SP 5076543210'),
 'laudo_06.txt': ref('apartamento','Rua das Palmeiras, 1.210, Recife/PE, CEP 50000-100', [('privativa',61,()),('total',84,())],2018,455000,'9.876','nao_informado','2025-04-15','Fernanda Lins, CREA 18001/PE'),
 'laudo_07.txt': ref('casa geminada','Rua Azul, 33, bairro Jardim Europa, Porto Alegre - RS', [('terreno',125,()),('construida',92.5,())],2011,372000,'66.504','onus_identificados','2025-04-20','Eng. Rafael Costa, CREA-RS 222333'),
 'laudo_08.txt': ref('loja térrea com sobreloja','Rua Sete de Setembro, 410, Salvador/BA', [('construida',118,())],None,910000,None,'nao_informado','2025-04-29','Luciana Prado - CNAI 12345'),
 'laudo_09.txt': ref('casa residencial','Alameda das Flores 88, Florianópolis/SC', [('terreno',420,()),('construida',198,())],2020,1080000,'12.909','sem_onus_identificados','2025-05-03','Eng. Thiago Martins, CREA-SC 7654321'),
 'laudo_10.txt': ref('apartamento','Condomínio Parque Norte, Brasília/DF, SQN 214, bloco C, apto 407', [('privativa',96.3,()),('total',127.6,())],2016,735000,'201.443','onus_identificados','2025-05-09','Denise Carvalho, CREA-DF 112233'),
 'laudo_11.txt': ref('terreno para incorporação','Rua Projetada 4, s/n, Santos, SP', [('terreno',1020,())],None,2450000,'88.710','nao_informado',None,'Marcos Vieira, Eng. Civil, CREA-SP 5099988776'),
 'laudo_12.txt': ref('casa de alto padrão','Rua das Bromélias, 500, Lago Sul, Brasília - DF', [('construida',285,()),('terreno',600,())],None,2180000,'54.122','onus_identificados','2025-05-23','Arq. Andréa Melo, CAU A445566-1'),
 'laudo_13.txt': ref('apartamento','Av. Brasil 1770, ap. 1201, Rio de Janeiro/RJ', [('util',112,()),('total',155,())],1987,1320000,'145.230','onus_cancelado','2025-05-30','Eduardo Sampaio, CNAI 67890'),
 'laudo_14.txt': ref('galpão industrial','Rodovia BR-116, km 12, Betim/MG', [('coberta',1450,()),('terreno',3000,())],None,3900000,'70.008','nao_informado','2025-06-05','Sérgio Tavares, CREA-MG 998877'),
 'laudo_15.txt': ref('casa','Rua Monte Verde, 19, Curitiba-PR', [('terreno',200,()),('construida',135,())],None,590000,'39.240','onus_identificados','2025-06-12','Patrícia Gomes, Eng. Civil, CREA-PR 123123'),
 'laudo_16.txt': ref('unidade comercial 14','Rua do Sol, 250, Campinas/SP', [('privativa',42,()),('total',67,())],2010,288000,'101.010','nao_informado','2025-06-19','Guilherme Rocha, CREA-SP 501010'),
 'laudo_17.txt': ref('apartamento residencial','Rua Harmonia, 44, Vila Madalena, São Paulo/SP', [('privativa',70,()),('total',None,(92,95))],2015,610000,'176.543','sem_onus_declarados_sem_certidao','2025-06-25','Marina Albuquerque, CREA-SP 5061234567'),
}
assert set(GABARITO_RASCUNHO) == set(TEXTOS), 'Referência e documentos não correspondem.'
print('Referência preenchida para', len(GABARITO_RASCUNHO), 'laudos. Revise os casos ambíguos antes de publicar métricas.')


Referência preenchida para 17 laudos. Revise os casos ambíguos antes de publicar métricas.


## 9. Métricas reprodutíveis e diferenças por campo

O escore principal é **acertos / 153** quando todos os 17 documentos foram extraídos. Se houver falha, a tabela mostra acertos, campos avaliados e campos esperados, sem esconder arquivos ausentes. Também registramos **preenchimentos indevidos** onde a referência usa `null`, acertos por campo e pendências de evidência. Métricas com esta referência provisória são exploratórias.


In [11]:
import re
import unicodedata

CAMPOS = ['tipo_imovel','endereco','areas','ano_construcao','valor_avaliacao_brl',
          'matricula','onus','data_vistoria','responsavel_tecnico']

def normalizar_texto(valor):
    if valor is None:
        return None
    sem_acento = ''.join(c for c in unicodedata.normalize('NFKD', str(valor).lower()) if not unicodedata.combining(c))
    return ' '.join(re.findall(r'[a-z0-9]+', sem_acento))

def normalizar_areas(areas):
    def uma(tipo, valor, conflitantes):
        return (tipo, None if valor is None else round(float(valor), 2),
                tuple(sorted(round(float(x), 2) for x in conflitantes)))
    if areas and isinstance(areas[0], Area):
        valores = [uma(a.tipo, a.valor_m2, a.valores_conflitantes_m2) for a in areas]
    else:
        valores = [uma(*a) for a in areas]
    return sorted(valores, key=lambda x: x[0])

def comparar(campo, previsto, esperado):
    if campo == 'areas':
        return normalizar_areas(previsto) == normalizar_areas(esperado)
    if campo == 'onus':
        return previsto.situacao == esperado
    if campo in ('tipo_imovel','endereco','matricula','responsavel_tecnico'):
        return normalizar_texto(previsto) == normalizar_texto(esperado)
    if campo == 'valor_avaliacao_brl':
        return (previsto is None and esperado is None) or (
            previsto is not None and esperado is not None and abs(previsto-esperado) <= 0.01)
    return previsto == esperado

linhas = []
for nome, gold in GABARITO_RASCUNHO.items():
    item = resultados.get(nome)
    for campo in CAMPOS:
        esperado = gold[campo]
        previsto = None if item is None else getattr(item, campo)
        acerto = False if item is None else comparar(campo, previsto, esperado)
        valor_exibido = (previsto.model_dump() if campo == 'onus' and previsto is not None else
                          [a.model_dump() for a in previsto] if campo == 'areas' and previsto is not None else previsto)
        linhas.append({'arquivo':nome,'campo':campo,'acerto':acerto,'extraido':item is not None,
                       'esperado':str(esperado),'obtido':str(valor_exibido),
                       'preenchimento_indevido':bool(item is not None and esperado is None and previsto is not None)})

comparacao = pd.DataFrame(linhas)
resumo_campo = comparacao.groupby('campo').agg(acertos=('acerto','sum'), avaliados=('extraido','sum'),
                                                esperados=('arquivo','size')).reset_index()
resumo_campo['taxa_nos_avaliados'] = resumo_campo.apply(
    lambda r: round(r.acertos/r.avaliados, 3) if r.avaliados else None, axis=1)
comparacao.to_csv(SAIDA/'comparacao_campos.csv', index=False, encoding='utf-8-sig')
print('Acertos:', int(comparacao.acerto.sum()), '/', len(comparacao),
      '| campos de laudos extraídos:', int(comparacao.extraido.sum()),
      '| preenchimentos indevidos:', int(comparacao.preenchimento_indevido.sum()))
display(resumo_campo)
display(comparacao[~comparacao.acerto].head(30))


Acertos: 134 / 153 | campos de laudos extraídos: 153 | preenchimentos indevidos: 0


,campo,acertos,avaliados,esperados,taxa_nos_avaliados
0,ano_construcao,15,17,17,0.882
1,areas,12,17,17,0.706
2,data_vistoria,17,17,17,1.000
3,endereco,14,17,17,0.824
4,matricula,13,17,17,0.765
5,onus,17,17,17,1.000
6,responsavel_tecnico,15,17,17,0.882
7,tipo_imovel,14,17,17,0.824
8,valor_avaliacao_brl,17,17,17,1.000


,arquivo,campo,acerto,extraido,esperado,obtido,preenchimento_indevido
5,laudo_01.txt,matricula,False,True,184.772,184.772 do 14º CRI de São Paulo,False
11,laudo_02.txt,areas,False,True,"[('construida', 146, ()), ('terreno', 250, ())]","[{'tipo': 'construida', 'valor_m2': None, 'val...",False
17,laudo_02.txt,responsavel_tecnico,False,True,"Carlos Henrique Moura, CAU A123456-7",Carlos Henrique Moura,False
18,laudo_03.txt,tipo_imovel,False,True,sala comercial 503,sala comercial,False
19,laudo_03.txt,endereco,False,True,"Edifício Horizonte, Rua do Comércio, 77, Curit...","Rua do Comércio, 77, Edifício Horizonte, sala ...",False
20,laudo_03.txt,areas,False,True,"[('util', 54.8, ()), ('comum', 18.2, ())]","[{'tipo': 'util', 'valor_m2': None, 'valores_c...",False
29,laudo_04.txt,areas,False,True,"[('terreno', 360, ())]","[{'tipo': 'terreno', 'valor_m2': None, 'valore...",False
37,laudo_05.txt,endereco,False,True,"Município de Campinas, UF SP","Campinas, SP",False
48,laudo_06.txt,ano_construcao,False,True,2018,None,False
50,laudo_06.txt,matricula,False,True,9.876,"9.876, 2º RGI do Recife",False


## 10. Resumo do que pode ser defendido na entrevista

- **Formato:** esquema obrigatório no pedido + validação Pydantic; falha de API/JSON é registrada e não vira dado inventado.
- **Correção:** referência por campo e comparação explícita (número, data, texto e tipo de ônus); `comparacao_campos.csv` permite inspecionar cada divergência. **Revise a referência assistida por IA para obter um gabarito humano.**
- **Ausência:** `null` com motivo em `alertas`; `nao_informado` para ônus sem informação; `[]` quando não houver áreas.
- **Contradição:** manter alternativas, deixar o valor resolvido como `null` e sinalizar revisão. O laudo 17 testa área total conflitante; o 13 testa ônus cancelado; o 11 distingue levantamento de vistoria.
- **Limites:** JSON estruturalmente válido não implica extração correta; evidência literal não prova que o valor foi interpretado adequadamente. A exatidão semântica de endereço, nome e classificação jurídica depende de revisão. Taxas só representam esta pequena amostra de 17 documentos sintéticos.

Anote no diário de bordo (Parte 4) o modelo usado, o prompt, correções que você fez após inspecionar `comparacao_campos.csv` e o tempo gasto. **Não altere o gabarito depois de olhar uma resposta apenas para elevar a pontuação**; justifique mudanças com o `.txt` original.
